In [1]:
from giskardpy_ros.python_interface.python_interface import GiskardWrapper
import rospy
from geometry_msgs.msg import PoseStamped, Point, Quaternion
from tf.transformations import quaternion_from_matrix
from giskardpy.model.world_config import WorldWithOmniDriveRobot
from giskardpy.casadi_wrapper import TransMatrix

In [2]:
rospy.init_node('test')

In [3]:
giskard = GiskardWrapper()

In [4]:
pose = PoseStamped()
pose.header.frame_id = 'map'
pose.pose.orientation.w = 1

giskard.world.add_urdf(name='dlr_kitchen', urdf=rospy.get_param('kitchen_description'), pose=pose)

DuplicateNameException: Group with name 'dlr_kitchen' already exists.

In [4]:
pose = PoseStamped()
pose.header.frame_id = 'map'
pose.pose.position = Point(1, 2, 0)
pose.pose.orientation = Quaternion(*quaternion_from_matrix([[0, -1, 0, 0],
                                                            [1, 0, 0, 0],
                                                            [0, 0, 1, 0],
                                                            [0, 0, 0, 1]]))
giskard.motion_goals.add_cartesian_pose(pose, 'base_link', 'map')
giskard.add_default_end_motion_conditions()
giskard.execute()

error: 
  type: ''
  msg: ''
trajectory: 
  header: 
    seq: 0
    stamp: 
      secs: 0
      nsecs:         0
    frame_id: ''
  joint_names: 
    - fl_caster_rotation_joint
    - fl_caster_l_wheel_joint
    - fl_caster_r_wheel_joint
    - fr_caster_rotation_joint
    - fr_caster_l_wheel_joint
    - fr_caster_r_wheel_joint
    - bl_caster_rotation_joint
    - bl_caster_l_wheel_joint
    - bl_caster_r_wheel_joint
    - br_caster_rotation_joint
    - br_caster_l_wheel_joint
    - br_caster_r_wheel_joint
    - torso_lift_joint
    - head_pan_joint
    - head_tilt_joint
    - laser_tilt_mount_joint
    - r_shoulder_pan_joint
    - r_shoulder_lift_joint
    - r_upper_arm_roll_joint
    - r_elbow_flex_joint
    - r_forearm_roll_joint
    - r_wrist_flex_joint
    - r_wrist_roll_joint
    - r_gripper_motor_slider_joint
    - r_gripper_motor_screw_joint
    - r_gripper_l_finger_joint
    - r_gripper_l_finger_joint
    - r_gripper_l_finger_joint
    - r_gripper_l_finger_joint
    - r_gripper_

In [5]:
from pyswip import Prolog, registerForeign

In [6]:
# second local giskard instance
urdf = open('pr2_local.urdf', 'r').read()
config = WorldWithOmniDriveRobot(urdf=urdf)
with config.world.modify_world():
    config.setup()
config.world.register_controlled_joints(config.world.movable_joint_names)

t = TransMatrix()
with config.world.modify_world():
    config.world.add_urdf(open('dlr_kitchen.urdf', 'r').read(), parent_link_name='map', pose=t)

In [7]:
import rospy
from trajectory_msgs.msg import JointTrajectory
from visualization_msgs.msg import Marker, MarkerArray
from std_msgs.msg import ColorRGBA
from geometry_msgs.msg import Point

marker_pub = rospy.Publisher("/trajectory_markers", MarkerArray, queue_size=10)

def joint_trajectory_to_marker_array(msg: JointTrajectory, frame_id: str = "map") -> MarkerArray:
    marker_array = MarkerArray()

    # Trajectory line marker
    line_marker = Marker()
    line_marker.header.frame_id = frame_id
    line_marker.header.stamp = rospy.Time.now()
    line_marker.ns = "trajectory"
    line_marker.id = 0
    line_marker.type = Marker.LINE_STRIP
    line_marker.action = Marker.ADD
    line_marker.scale.x = 0.02  # Line thickness
    line_marker.color = ColorRGBA(1.0, 0.0, 0.0, 1.0)  # Red color
    line_marker.pose.orientation.w = 1.0  # Identity quaternion (no rotation)

    # Sphere markers for each trajectory point
    point_markers = []

    for idx, point in enumerate(msg.points):
        if len(point.positions) < 3:
            rospy.logwarn("Trajectory points must have at least 3 values (x, y, z). Skipping point.")
            continue
        x, y, z = point.positions[:3]  # Assume first 3 positions are x, y, z
        # Add to the line strip
        line_marker.points.append(Point(x, y, z))
    # Add all markers to the MarkerArray
    marker_array.markers.append(line_marker)

    return marker_array

In [10]:
# TODO add a function that uses giskard only for the articulated environment fo find a general motion for opening/closing
prolog = Prolog()

def execute():
    giskard.motion_goals.allow_all_collisions()
    giskard.execute()

def add_init_pose():
    pose = PoseStamped()
    pose.header.frame_id = 'map'
    pose.pose.position = Point(1, 2, 0)
    pose.pose.orientation = Quaternion(*quaternion_from_matrix([[0, -1, 0, 0],
                                                                [1, 0, 0, 0],
                                                                [0, 0, 1, 0],
                                                                [0, 0, 0, 1]]))
    giskard.motion_goals.add_cartesian_pose(pose, 'base_link', 'map')
    giskard.add_default_end_motion_conditions()

def traj_open_container(motion, joint, goalState, handle, feedback):
    if str(motion) != 'envJointGoal':
        return False
    if isinstance(joint, bytes):
            joint = joint.decode('utf-8')
    if isinstance(handle, bytes):
            handle = joint.decode('utf-8')
    # marker_pub.publish(MarkerArray())

    giskard.motion_goals.add_joint_position_return_traj(goal_state={str(joint): float(goalState)}, root_link='map', traj_frame=str(handle))
    mon2 = giskard.monitors.add_joint_position(goal_state={str(joint): float(goalState)})

    loc = giskard.monitors.add_local_minimum_reached()
    giskard.monitors.add_end_motion(mon2)
    giskard.monitors.add_cancel_motion(loc, Exception('local min'))
    res = giskard.projection()

    marker_pub.publish(joint_trajectory_to_marker_array(res.trajectory))
    reason = giskard.get_end_motion_reason(res)

    if len(reason) == 0:
        feedback.unify('success')
    else:
        feedback.unify(str(reason))
    return True

def add_open_container(motion, joint, goalState, handle, gripper):
    if str(motion) != 'envJointGoal':
        return False
    if isinstance(joint, bytes):
            joint = joint.decode('utf-8')
    pose = PoseStamped()
    pose.header.frame_id = str(handle)
    pose.pose.position = Point(0, 0, 0)
    pose.pose.orientation.w = 1

    # mon1 = giskard.monitors.add_cartesian_pose(root_link='base_link', tip_link='r_gripper_tool_frame', goal_pose=pose, position_threshold=0.03)
    giskard.motion_goals.add_cartesian_pose(pose, str(gripper), 'map')
    giskard.add_default_end_motion_conditions()
    giskard.motion_goals.allow_all_collisions()
    giskard.execute()

    giskard.motion_goals.add_open_container(tip_link=str(gripper), environment_link=str(handle),
                                            goal_joint_state=float(goalState))
    mon2 = giskard.monitors.add_joint_position(goal_state={str(joint): float(goalState)})

    giskard.monitors.add_end_motion(mon2)
    giskard.monitors.add_max_trajectory_length(20)

def giskard_project_eval(feedback):
    giskard.motion_goals.allow_all_collisions()
    # giskard.monitors.add_check_trajectory_length(40)
    result = giskard.projection()
    reason = giskard.get_end_motion_reason(result)
    # print(reason)
    if len(reason) == 0:
        feedback.unify('success')
    else:
        feedback.unify(str(reason))
    return True

registerForeign(add_init_pose, arity=0)
registerForeign(add_open_container, arity=5)
registerForeign(execute, arity=0)
registerForeign(giskard_project_eval, arity=1)
registerForeign(traj_open_container, arity=5)

def execute_motion_envJointGoal(motion, joint, goalState, handle, gripper):
    if str(motion) != 'envJointGoal':
        return False
    if isinstance(joint, bytes):
            joint = joint.decode('utf-8')
    pose = PoseStamped()
    pose.header.frame_id = 'map'
    pose.pose.position = Point(1, 2, 0)
    pose.pose.orientation = Quaternion(*quaternion_from_matrix([[0, -1, 0, 0],
                                                                [1, 0, 0, 0],
                                                                [0, 0, 1, 0],
                                                                [0, 0, 0, 1]]))
    giskard.motion_goals.add_cartesian_pose(pose, 'base_link', 'map')
    giskard.add_default_end_motion_conditions()
    giskard.execute()

    pose = PoseStamped()
    pose.header.frame_id = str(handle)
    pose.pose.position = Point(0, 0, 0)
    pose.pose.orientation.w = 1

    # mon1 = giskard.monitors.add_cartesian_pose(root_link='base_link', tip_link='r_gripper_tool_frame', goal_pose=pose, position_threshold=0.03)
    giskard.motion_goals.add_cartesian_pose(pose, str(gripper), 'map')
    giskard.add_default_end_motion_conditions()
    giskard.execute()

    giskard.motion_goals.add_open_container(tip_link=str(gripper), environment_link=str(handle),
                                            goal_joint_state=float(goalState))
    mon2 = giskard.monitors.add_joint_position(goal_state={str(joint): float(goalState)})

    giskard.monitors.add_end_motion(mon2)
    giskard.monitors.add_max_trajectory_length(20)
    result = giskard.projection()
    reason = giskard.get_end_motion_reason(result)
    if len(reason) == 0:
        return True
    return False


registerForeign(execute_motion_envJointGoal, arity=5)


def hasArticulation(joint, link):
    tip = config.world.search_for_link_name(str(link))
    result = config.world.get_movable_parent_joint(tip).short_name
    if result:
        joint.unify(str(result))
        return True
    return False


def partOf(link1, link2):
    l1 = config.world.search_for_link_name(str(link1))
    l2 = config.world.search_for_link_name(str(link2))
    results = config.world.get_links_in_branch_of_link(l1)
    return l2 in results


registerForeign(hasArticulation, arity=2)
registerForeign(partOf, arity=2)

prolog.consult("kb.pl")

In [12]:
# results = list(prolog.query(
#     "container(Container), handle(Handle), hasArticulation(Joint, Handle), gripper(Gripper), openState(Container, GoalState),"
#     "partOf(Handle, Container), canExecute(envJointGoal, Joint, GoalState, Handle, Gripper)."))
# results = list(prolog.query('container(C), handle(H), hasArticulation(J, H), partOf(H, C).'))
results = list(prolog.query(
                            'taskRequest(open_fridge, StateChange), '
                            'causes(Motion, StateChange, MotionParam, FeedbackTwo),'
                            'canPerform(Robot, Motion, MotionParam, Feedback).'
                            ))

for result in results:
    print(result)
print(len(results))

Exception ignored on calling ctypes callback function: <function _foreignWrapper.<locals>.wrapper at 0x7fc36b033af0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.8/dist-packages/pyswip/easy.py", line 527, in wrapper
    r = fun(*args)
  File "/tmp/ipykernel_235315/1789203066.py", line 66, in add_open_container
AttributeError: 'MonitorWrapper' object has no attribute 'add_max_trajectory_length'
Exception ignored on calling ctypes callback function: <function _foreignWrapper.<locals>.wrapper at 0x7fc36af58670>
Traceback (most recent call last):
  File "/usr/local/lib/python3.8/dist-packages/pyswip/easy.py", line 527, in wrapper
    r = fun(*args)
  File "/tmp/ipykernel_235315/1789203066.py", line 71, in giskard_project_eval
  File "/home/huerkamp/workspace/giskard_lib_ws/src/giskardpy_ros/src/giskardpy_ros/python_interface/python_interface.py", line 2340, in projection
    return self._send_action_goal(MoveGoal.PROJECTION, wait)
  File "/home/huerkamp/workspace/giska

PrologError: Caused by: 'taskRequest(open_fridge, StateChange), causes(Motion, StateChange, MotionParam, FeedbackTwo),canPerform(Robot, Motion, MotionParam, Feedback).'. Returned: 'error(domain_error(foreign_return_value, 27), context(/(giskard_project_eval, 1), _428))'.

In [9]:
results = list(prolog.query(
                            'taskRequest(X, StateChange),'
                            'causes(Motion, StateChange, MotionParam, Feedback).'
                            ))

for result in results:
    print(result)
print(len(results))

[positions: [0.9182801602759469, 3.5831840861015376, 0.9438999891281128, 1.0, 0.0]
velocities: [0.0, 0.0, 0.0, 0.0, 0.0]
accelerations: [0.0, 0.0, 0.0, 0.0, 0.0]
effort: []
time_from_start: 
  secs: 0
  nsecs:         0, positions: [0.9182801602759469, 3.5831840861015376, 0.9438999891281128, 1.0, 0.0]
velocities: [0.0, 0.0, 0.0, 0.0, 0.0]
accelerations: [0.0, 0.0, 0.0, 0.0, 0.0]
effort: []
time_from_start: 
  secs: 0
  nsecs:         0, positions: [0.9182801602759469, 3.5831840861015376, 0.9438999891281128, 1.0, 0.0]
velocities: [0.0, 0.0, 0.0, 0.0, 0.0]
accelerations: [0.0, 0.0, 0.0, 0.0, 0.0]
effort: []
time_from_start: 
  secs: 0
  nsecs:         0, positions: [0.9182801602759469, 3.5831840861015376, 0.9438999891281128, 1.0, 0.0]
velocities: [0.0, 0.0, 0.0, 0.0, 0.0]
accelerations: [0.0, 0.0, 0.0, 0.0, 0.0]
effort: []
time_from_start: 
  secs: 0
  nsecs:         0, positions: [0.9182801602759469, 3.5831840861015376, 0.9438999891281128, 1.0, 0.0]
velocities: [0.0, 0.0, 0.0, 0.0, 0.0]

In [55]:
results = list(prolog.query(
                            'taskRequest(open_fridge, StateChange).'
                            ))

for result in results:
    print(result)
print(len(results))

{'StateChange': 'open(fridge)'}
1


In [29]:
# Todo: use knowrob to detect container and handles
giskard.motion_goals.add_joint_position_return_traj(goal_state={'fridge_door_joint': 0.8}, root_link='map', traj_frame='fridge_door_handle')
mon2 = giskard.monitors.add_joint_position(goal_state={'fridge_door_joint': 0.8})
loc = giskard.monitors.add_local_minimum_reached()
giskard.monitors.add_end_motion(mon2)
giskard.monitors.add_cancel_motion(loc, Exception('local min'))
res = giskard.execute()


In [38]:
pose = PoseStamped()
pose.header.frame_id = 'map'
pose.pose.orientation.w = 1

giskard.world.add_urdf(name='dlr_kitchen', urdf=rospy.get_param('kitchen_description'), pose=pose)

error: 
  type: ''
  msg: ''

In [13]:
giskard.get_end_motion_reason(res)

{}

In [29]:
from giskardpy.god_map import god_map

In [30]:
res

error: 
  type: ''
  msg: ''
trajectory: 
  header: 
    seq: 0
    stamp: 
      secs: 0
      nsecs:         0
    frame_id: "base_link"
  joint_names: 
    - trajectory|0
    - trajectory|1
    - trajectory|2
    - trajectory|3
    - trajectory
  points: 
    - 
      positions: [0.918280160845133, 3.583184081095569, 0.9438999891281128, 1.0, 0.0]
      velocities: [0.0, 0.0, 0.0, 0.0, 0.0]
      accelerations: [0.0, 0.0, 0.0, 0.0, 0.0]
      effort: []
      time_from_start: 
        secs: 0
        nsecs:         0
    - 
      positions: [0.9183717154369699, 3.582383813549573, 0.9438999891281128, 1.0, 0.0]
      velocities: [0.0073243673469569615, -0.0640214036796749, 0.0, 0.0, 0.0]
      accelerations: [0.0, 0.0, 0.0, 0.0, 0.0]
      effort: []
      time_from_start: 
        secs: 0
        nsecs:         0
    - 
      positions: [0.9186530470628717, 3.579983784695213, 0.9438999891281128, 1.0, 0.0]
      velocities: [0.02250653007213721, -0.1920023083487976, 0.0, 0.0, 0.0]
    

In [25]:
import rospy
from trajectory_msgs.msg import JointTrajectory
from visualization_msgs.msg import Marker, MarkerArray
from std_msgs.msg import ColorRGBA
from geometry_msgs.msg import Point

marker_pub = rospy.Publisher("/trajectory_markers", MarkerArray, queue_size=10)

def joint_trajectory_to_marker_array(msg: JointTrajectory, frame_id: str = "map") -> MarkerArray:
    marker_array = MarkerArray()

    # Trajectory line marker
    line_marker = Marker()
    line_marker.header.frame_id = frame_id
    line_marker.header.stamp = rospy.Time.now()
    line_marker.ns = "trajectory"
    line_marker.id = 0
    line_marker.type = Marker.LINE_STRIP
    line_marker.action = Marker.ADD
    line_marker.scale.x = 0.02  # Line thickness
    line_marker.color = ColorRGBA(1.0, 0.0, 0.0, 1.0)  # Red color
    line_marker.pose.orientation.w = 1.0  # Identity quaternion (no rotation)

    # Sphere markers for each trajectory point
    point_markers = []

    for idx, point in enumerate(msg.points):
        if len(point.positions) < 3:
            rospy.logwarn("Trajectory points must have at least 3 values (x, y, z). Skipping point.")
            continue
        x, y, z = point.positions[:3]  # Assume first 3 positions are x, y, z
        # Add to the line strip
        line_marker.points.append(Point(x, y, z))
    # Add all markers to the MarkerArray
    marker_array.markers.append(line_marker)

    return marker_array




In [32]:

# Publisher for MarkerArray
marker_pub = rospy.Publisher("/trajectory_markers", MarkerArray, queue_size=10)


In [35]:
marker_pub.publish(joint_trajectory_to_marker_array(res.trajectory))

In [39]:
# Todo: use knowrob to detect container and handles
marker_pub.publish(MarkerArray())
##########
goal = 0.9
giskard.motion_goals.add_joint_position_return_traj(goal_state={'fridge_door_joint': goal}, root_link='map', traj_frame='fridge_door_handle')
mon2 = giskard.monitors.add_joint_position(goal_state={'fridge_door_joint': goal})
loc = giskard.monitors.add_local_minimum_reached()
giskard.monitors.add_end_motion(mon2)
giskard.monitors.add_cancel_motion(loc, Exception('local min'))
res = giskard.projection()
###########
marker_pub.publish(joint_trajectory_to_marker_array(res.trajectory))

In [42]:
marker_pub.publish(MarkerArray())